# Train YOLOv8s vessel detector on SMD (video-level split)

**No Google Drive mount.** You upload `smd_yolo.zip` from your laptop, train, and download the results.

Before you start: **Runtime → Change runtime type → T4 GPU**.

Colab's disk is wiped when the session ends — download the results (last cell) before closing the tab.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Install Ultralytics

In [ ]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

## 3. Upload the dataset zip from your laptop

Run the cell and pick `data/smd_yolo.zip` (made by `training/prepare_smd_yolo.py`).
If the upload stalls for a large file, drag the zip into the **Files** panel on the left instead, then run the next cell.

In [ ]:
import os
from google.colab import files

if not os.path.exists("/content/smd_yolo.zip"):
    uploaded = files.upload()          # choose smd_yolo.zip
    name = next(iter(uploaded))
    if name != "smd_yolo.zip":
        os.rename(name, "smd_yolo.zip")
print("zip size:", round(os.path.getsize("/content/smd_yolo.zip") / 1e6), "MB")

## 4. Unzip and point data.yaml at the Colab folder

In [ ]:
!rm -rf /content/smd_yolo && unzip -q /content/smd_yolo.zip -d /content
import json, yaml

cfg = {
    "path": "/content/smd_yolo",
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {0: "vessel"},
}
with open("/content/smd_yolo/data.yaml", "w") as f:
    yaml.safe_dump(cfg, f)

meta = json.load(open("/content/smd_yolo/splits.json"))
for split, s in meta["stats"].items():
    print(f"{split:5s} videos {s['videos']:3d} | images {s['images']:5d} | boxes {s['boxes']:6d}")
print("test videos:", meta["splits"]["test"])

## 5. Train

Settings, chosen on purpose:
- `yolov8s.pt` — same model size as before, so the laptop (RTX 2050, 4 GB) can still run it.
- `imgsz=960` — larger than the default 640 because many vessels are tiny; the backend also runs at 960.
- `seed=0, deterministic=True` — reproducible runs for the paper.
- `patience=20` — stop early if validation mAP stops improving.

**Optional — fine-tune your old model instead:** upload `smd_yolov8s_vessel_fresh_best.pt` and set `START = "/content/smd_yolov8s_vessel_fresh_best.pt"`.
Only do this if the old model was trained on a split that does not include this run's val/test videos.

In [ ]:
from ultralytics import YOLO

START = "yolov8s.pt"
RUN_NAME = "smd_v8s_videosplit"

model = YOLO(START)
model.train(
    data="/content/smd_yolo/data.yaml",
    imgsz=960,
    epochs=100,
    patience=20,
    batch=16,
    seed=0,
    deterministic=True,
    project="/content/runs",
    name=RUN_NAME,
    exist_ok=True,
)

## 6. Evaluate on the held-out TEST videos (never seen in training or validation)

In [ ]:
best = YOLO(f"/content/runs/{RUN_NAME}/weights/best.pt")
metrics = best.val(data="/content/smd_yolo/data.yaml", split="test", imgsz=960, batch=16,
                   project="/content/runs", name=f"{RUN_NAME}_test", exist_ok=True)

result = {
    "precision": round(float(metrics.box.mp), 4),
    "recall": round(float(metrics.box.mr), 4),
    "mAP50": round(float(metrics.box.map50), 4),
    "mAP50_95": round(float(metrics.box.map), 4),
}
print(result)
import json
json.dump(result, open(f"/content/runs/{RUN_NAME}/test_metrics.json", "w"), indent=2)

## 7. Download the results to your laptop (do this before closing Colab)

In [ ]:
import shutil
from google.colab import files

run = f"/content/runs/{RUN_NAME}"
keep = "/content/results_export"
shutil.rmtree(keep, ignore_errors=True)
shutil.copytree(f"{run}/weights", f"{keep}/weights")
for name in ["results.csv", "args.yaml", "test_metrics.json", "results.png", "confusion_matrix.png", "BoxPR_curve.png"]:
    try:
        shutil.copy(f"{run}/{name}", keep)
    except FileNotFoundError:
        pass
shutil.copy("/content/smd_yolo/splits.json", keep)
shutil.make_archive("/content/results", "zip", keep)
files.download("/content/results.zip")

## 8. On your laptop

```bash
cd ~/Desktop/martime_tracking_system
unzip ~/Downloads/results.zip -d training/runs/smd_v8s_videosplit
cp training/runs/smd_v8s_videosplit/weights/best.pt backend/models/best.pt
```
Record the test metrics in `training/README.md` (Results table).